# Joint lexical + semantic grounding - dataset benchmark

**Author**: kj  
**Approach**: race the fusion wirings that compose the OpenVINO semantic cascade with the lexical tier, on the 2,752-claim verified gold, and decide which to freeze into the `semantic` switch.

The lexical manifold (effort=high) is cheap and torch-free; the semantic cascade (bge-m3 pre-filter -> bge-reranker + mDeBERTa-NLI) is stronger but heavy. The `semantic` switch composes them by **escalation**: the lexical verdict decides whenever its win is clear, and only the uncertain band escalates to the cascade. This notebook measures three ways of fusing them:

- **W1 escalation** - lexical decides; the uncertain band + cross-lingual blocks escalate (the shipped design)
- **W2 always-both** - one joint logistic over every claim
- **W3 reuse-seam** - lexical + cosine + NLI only, no reranker (lower bound)

All cascade signals come from the cached int8 pair scores, so the run is deterministic and GPU-free (OOF 5-fold, seed 42). Metrics are aggregate only - no per-claim text is shown.

## Imports

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")  # CPU only; the cascade signals are cached

# the experiment modules import siblings flat -> put the leg dir on sys.path
LEG = Path.cwd().parents[1] / "experiments" / "grounding-semantic"
sys.path.insert(0, str(LEG))

import joint_wirings as jw

## Configuration

Cached signals live under the gitignored `private-rag-forensics/`. The lexical P is read from `lexical_verdict_high.npz` (built once by `joint_wirings.lexical_pass`); the cascade scores from the int8 pair cache.

In [2]:
FORENSICS = LEG / "private-rag-forensics"
assert FORENSICS.exists(), (
    "private-rag-forensics/ absent - this benchmark needs the gitignored gold + cached "
    "scores present locally (it does not run on CI)."
)
print("leg:", LEG)
print("lexical cache:", (FORENSICS / "lexical_verdict_high.npz").exists())
print("pair cache:", (FORENSICS / "model_scores" / "pairs" / "full_pairs.npz").exists())

leg: /home/lab/workspace/private/ai-assistants/claude-code-plugins/groundrails/experiments/grounding-semantic
lexical cache: True
pair cache: True


## Load the per-claim signals

`load_signals` aligns, per claim: the lexical verdict P / fired / contradiction / blocked, the cascade reranker / NLI-entailment / NLI-contradiction maxes, and the bi-encoder cosine max - plus the gold label and language.

In [3]:
s = jw.load_signals()
y = s["y"]
n = len(y)
n_hall = int((y == 0).sum())
n_blocked = int(s["lex_blocked"].sum())
print(f"{n} claims | {n_hall} hallucination / {n - n_hall} supported "
      f"(base rate {n_hall / n:.0%}) | {n_blocked} cross-lingual blocked")

2752 claims | 786 hallucination / 1966 supported (base rate 29%) | 42 cross-lingual blocked


## Run the wirings

Each wiring is scored out-of-fold; the threshold sits at the macro-F1 optimum. macro-F1 weights the hallucination- and supported-class F1 equally; FP = supported wrongly flagged, FN = hallucination missed.

In [4]:
rows = [jw.lexical_only(s), jw.w3_reuse_seam(s), jw.w2_always_both(s), jw.w1_escalation(s)]
base = rows[0]

df = pd.DataFrame([
    {
        "wiring": r["name"],
        "macro-F1": round(r["macro"], 3),
        "FP": r["fp"],
        "FN": r["fn"],
        "FP+FN": r["fp"] + r["fn"],
        "escalation": "-" if r is base else f"{r['escalation']:.0%}",
        "d vs lexical": "" if r is base else f"{r['macro'] - base['macro']:+.3f}",
    }
    for r in rows
])
df

,wiring,macro-F1,FP,FN,FP+FN,escalation,d vs lexical
0,lexical-only (high),0.759,514,97,611,-,
1,"W3 reuse-seam {lex,cos,nli}",0.826,155,225,380,100%,+0.067
2,W2 always-both joint,0.822,243,167,410,100%,+0.063
3,W1 escalation cascade,0.822,247,164,411,90%,+0.063


## Escalation cost lever (W1)

W1 is the only wiring with a cost lever: the escalation band controls how many claims pay for the cascade. The cheap lexical verdict resolves the rest.

In [5]:
w1 = next(r for r in rows if r["name"].startswith("W1"))
print(f"W1 escalation: macro-F1 {w1['macro']:.3f}, FP {w1['fp']}, FN {w1['fn']}")
print(f"  escalation band [{w1['band'][0]:.2f}, {w1['band'][1]:.2f}] "
      f"-> {w1['escalation']:.0%} of claims escalate to the cascade")
print(f"  the remaining {1 - w1['escalation']:.0%} resolve on the lexical verdict alone")

W1 escalation: macro-F1 0.822, FP 247, FN 164
  escalation band [0.13, 0.99] -> 90% of claims escalate to the cascade
  the remaining 10% resolve on the lexical verdict alone


## Conclusions

- **The semantic switch is worth it.** Every fusion lifts macro-F1 by **+0.06-0.07** over lexical-only - the cascade recovers the supported claims the conservative lexical manifold over-flags (its high FP).
- **The three wirings are statistically tied** (macro-F1 0.822-0.826 at n=2,752). The reranker (W2/W1) buys hallucination recall (lower FN); the reuse-seam (W3) trades that for fewer false flags (lower FP). macro-F1 cannot separate them.
- **W1 escalation is the shipped tier.** It is the requested design (lexical-first, escalate the uncertain band), carries the best hallucination recall, and is the only wiring with a cost lever. Its frozen joint head + escalation band live in `calibration.semantic` (`reports/semantic_tier_block.yaml`).
- **The lever has headroom.** The shipped high manifold over-flags supported claims, so the optimiser picks a wide escalation band here. A better-calibrated lexical gate would narrow the band and cut the cascade share at the same quality - the next optimisation.

Report: `reports/grounding_joint_wirings.md`. Regenerate end-to-end with `python experiments/grounding-semantic/joint_wirings.py`.